# Daily OLS residual raw spatial semivariogram, implied-spatial x1 smooth comparison, 2024 July days 1-10

This notebook is a narrow x1-only version of `daily_spatial_semivariogram_residual_ols_raw_three_model_2022_2025_052526.ipynb`.

Scope:
- year: 2024
- month: July
- days: 1-10
- empirical curve: raw spatial semivariogram of OLS residual maps, same mean removal as the reference notebook
- fitted curves: implied-spatial ST Vecchia x1 parameters only
- smooth values: 0.3 and 0.5
- default variant: `nugget_free`

No x8/x4/x2 parameter rows are used in this notebook.


In [ ]:
import os
import pickle
from pathlib import Path
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns
from scipy.special import gamma as gamma_fn, kv
from IPython.display import display

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path('/Users/joonwonlee/Documents/GEMS_TCO-1')
DATA_ROOT = Path('/Users/joonwonlee/Documents/GEMS_DATA')

YEAR = 2024
MONTH = 7
DAYS = list(range(1, 11))
LAT_RANGE = [-3.0, 2.0]
LON_RANGE = [121.0, 131.0]
MEAN_MODEL = 'hour_dummy_plus_lat_lon_ols'
NORMALIZE_EMPIRICAL = False
Y_LIMIT = (0, 20)

RESOLUTION_LABEL = 'x1'
VARIANT = 'nugget_free'
PRIMARY_PARAM_ROOT = PROJECT_ROOT / 'outputs/day/real_data/implied_spatial_spectral_corridor_4x4_lag643_lat-3to7_lon121to131_052126'
REFIT_PARAM_ROOT = PROJECT_ROOT / 'outputs/day/real_data/implied_spatial_x1_refit_smooth0p3_2024_july_days02_10_052926'
SMOOTH_SOURCES = {
    0.3: [
        REFIT_PARAM_ROOT / 'smooth_0p3/2024_07/daily_csv',
        PRIMARY_PARAM_ROOT / 'smooth_0p3/2024_07/daily_csv',
    ],
    0.5: [PRIMARY_PARAM_ROOT / 'smooth_0p5/2024_07/daily_csv'],
}

# Same positive univariate lag grid used by the empirical semivariogram scripts.
# Zero spatial lag is deliberately excluded for pure spatial semivariograms.
LAT_LAGS = np.concatenate([[0.044, 0.132], np.arange(0.176, 2.3, 0.044 * 5)])
LON_LAGS = np.concatenate([[0.063, 0.126], np.arange(0.18, 2.0, 0.063 * 3)])
LAT_LAGS = np.round(LAT_LAGS, 3)
LON_LAGS = np.round(LON_LAGS, 3)

OUTPUT_DIR = PROJECT_ROOT / 'plots/directional_semivariograms/daily_spatial_semivariograms_residual_raw_implied_spatial_x1_smooth03_05_2024_july10_052926'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output directory: {OUTPUT_DIR}')
print(f'Mean model: {MEAN_MODEL}')
print(f'Using only resolution_label={RESOLUTION_LABEL}, variant={VARIANT}')
print('Parameter source directories:')
for smooth, dirs in SMOOTH_SOURCES.items():
    print(f'  smooth={smooth}:')
    for d in dirs:
        print(f'    {d}')
print(f'Normalize empirical residual semivariogram: {NORMALIZE_EMPIRICAL}')
print('lat lags:', LAT_LAGS)
print('lon lags:', LON_LAGS)


In [ ]:
def load_x1_estimates_for_smooth(smooth: float, daily_csv_dirs: list[Path]) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    status_rows = []
    for day_num in DAYS:
        date_str = f'{YEAR}{MONTH:02d}{day_num:02d}'
        base_status = {
            'smooth': smooth,
            'year': YEAR,
            'month': MONTH,
            'day_num': day_num,
            'date_str': date_str,
        }
        candidates = []
        for source_i, daily_csv_dir in enumerate(daily_csv_dirs):
            path = daily_csv_dir / f'{date_str}_st_fits.csv'
            if not path.exists():
                status_rows.append({**base_status, 'source_priority': source_i, 'source_file': str(path), 'status': 'missing_file', 'error': 'file not found'})
                continue

            raw = pd.read_csv(path)
            sub = raw[
                raw['resolution_label'].astype(str).eq(RESOLUTION_LABEL)
                & raw['variant'].astype(str).eq(VARIANT)
            ].copy()
            if sub.empty:
                status_rows.append({**base_status, 'source_priority': source_i, 'source_file': str(path), 'status': 'missing_x1_row', 'error': f'no {RESOLUTION_LABEL}/{VARIANT} row'})
                continue

            row = sub.iloc[0].copy()
            status = str(row.get('status', '')).lower()
            status_rows.append({
                **base_status,
                'source_priority': source_i,
                'source_file': str(path),
                'status': row.get('status', np.nan),
                'error': row.get('error', np.nan),
                'resolution_label': row.get('resolution_label', np.nan),
                'variant': row.get('variant', np.nan),
                'lag0_block_count': row.get('lag0_block_count', np.nan),
                'lag1_block_count': row.get('lag1_block_count', np.nan),
                'lag2_block_count': row.get('lag2_block_count', np.nan),
                'n_clusters': row.get('n_clusters', np.nan),
            })
            candidates.append((status == 'ok', source_i, path, row))

        ok_candidates = [c for c in candidates if c[0]]
        if not ok_candidates:
            continue
        _, source_i, path, row = sorted(ok_candidates, key=lambda x: x[1])[0]

        rows.append({
            'model': f'nu={smooth:g} {RESOLUTION_LABEL} {VARIANT}',
            'smooth': smooth,
            'year': YEAR,
            'month': MONTH,
            'day_num': day_num,
            'date_str': date_str,
            'day': f'{YEAR}-{MONTH:02d}-{day_num:02d}',
            'resolution_label': row.get('resolution_label', RESOLUTION_LABEL),
            'variant': row.get('variant', VARIANT),
            'sigma': row.get('est_sigmasq', np.nan),
            'range_lat': row.get('est_range_lat', np.nan),
            'range_lon': row.get('est_range_lon', np.nan),
            'range_time': row.get('est_range_time', np.nan),
            'advec_lat': row.get('est_advec_lat', np.nan),
            'advec_lon': row.get('est_advec_lon', np.nan),
            'nugget': row.get('est_nugget', np.nan),
            'loss': row.get('loss', np.nan),
            'time': row.get('total_s', np.nan),
            'lag0_block_count': row.get('lag0_block_count', np.nan),
            'lag1_block_count': row.get('lag1_block_count', np.nan),
            'lag2_block_count': row.get('lag2_block_count', np.nan),
            'n_clusters': row.get('n_clusters', np.nan),
            'source_priority': source_i,
            'source_file': str(path),
        })

    return pd.DataFrame(rows), pd.DataFrame(status_rows)


estimate_frames = []
status_frames = []
for smooth, source_dirs in SMOOTH_SOURCES.items():
    est, stat = load_x1_estimates_for_smooth(smooth, source_dirs)
    estimate_frames.append(est)
    status_frames.append(stat)

estimates = pd.concat(estimate_frames, ignore_index=True) if estimate_frames else pd.DataFrame()
estimate_status = pd.concat(status_frames, ignore_index=True) if status_frames else pd.DataFrame()

numeric_cols = ['smooth', 'year', 'month', 'day_num', 'sigma', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'nugget', 'loss', 'time']
for col in numeric_cols:
    if col in estimates.columns:
        estimates[col] = pd.to_numeric(estimates[col], errors='coerce')

param_cols = [
    'year', 'day', 'model', 'smooth', 'resolution_label', 'variant', 'sigma', 'range_lat', 'range_lon',
    'range_time', 'advec_lat', 'advec_lon', 'nugget', 'loss', 'time',
    'lag0_block_count', 'lag1_block_count', 'lag2_block_count', 'n_clusters',
]
param_table = estimates[param_cols].round(4) if not estimates.empty else pd.DataFrame(columns=param_cols)
status_csv = OUTPUT_DIR / 'x1_parameter_status_smooth03_05_2024_07_days01_10.csv'
param_csv = OUTPUT_DIR / 'x1_parameter_estimates_smooth03_05_2024_07_days01_10.csv'
estimate_status.to_csv(status_csv, index=False)
param_table.to_csv(param_csv, index=False, float_format='%.4f')
print(f'Saved {status_csv}')
print(f'Saved {param_csv}')
print('Loaded OK x1 parameter rows:')
display(param_table)
print('Parameter status by smooth/day:')
display(estimate_status[['smooth', 'day_num', 'status', 'resolution_label', 'variant', 'lag0_block_count', 'lag1_block_count', 'lag2_block_count', 'n_clusters', 'error']])


In [ ]:
DATA_CACHE = {}
GRID_CACHE = {}
RESIDUAL_GRID_CACHE = {}


def load_year_raw_map(year: int) -> Dict[str, pd.DataFrame]:
    if year not in DATA_CACHE:
        path = DATA_ROOT / f'pickle_{year}' / f'tco_grid_{str(year)[2:]}_{MONTH:02d}.pkl'
        with open(path, 'rb') as f:
            DATA_CACHE[year] = pickle.load(f)
        print(f'Loaded {len(DATA_CACHE[year])} hourly maps for {year}-{MONTH:02d}')
    return DATA_CACHE[year]


def build_grid_meta(year: int):
    if year in GRID_CACHE:
        return GRID_CACHE[year]
    raw = load_year_raw_map(year)
    first_key = sorted(raw)[0]
    df = raw[first_key].copy()
    mask = (
        df['Latitude'].between(LAT_RANGE[0], LAT_RANGE[1])
        & df['Longitude'].between(LON_RANGE[0], LON_RANGE[1])
    ).to_numpy()
    sub = df.loc[mask].copy()
    lats = np.sort(sub['Latitude'].round(6).unique())
    lons = np.sort(sub['Longitude'].round(6).unique())
    lat_to_i = {float(v): i for i, v in enumerate(lats)}
    lon_to_j = {float(v): j for j, v in enumerate(lons)}
    lat_idx = sub['Latitude'].round(6).map(lat_to_i).to_numpy(dtype=int)
    lon_idx = sub['Longitude'].round(6).map(lon_to_j).to_numpy(dtype=int)
    meta = {
        'mask': mask,
        'lats': lats,
        'lons': lons,
        'lat_idx': lat_idx,
        'lon_idx': lon_idx,
        'lat_step': float(np.nanmedian(np.diff(lats))),
        'lon_step': float(np.nanmedian(np.diff(lons))),
    }
    GRID_CACHE[year] = meta
    print(f'{year}: grid shape={len(lats)}x{len(lons)}, lat_step={meta["lat_step"]:.4f}, lon_step={meta["lon_step"]:.4f}')
    return meta


def keys_for_day(raw_map: Dict[str, pd.DataFrame], day: int):
    token = f'day{int(day):02d}'
    return [k for k in sorted(raw_map) if token in k]


def compute_day_residual_grids(year: int, day: int):
    cache_key = (year, day)
    if cache_key in RESIDUAL_GRID_CACHE:
        return RESIDUAL_GRID_CACHE[cache_key]

    raw = load_year_raw_map(year)
    meta = build_grid_meta(year)
    day_keys = keys_for_day(raw, day)[:8]
    if len(day_keys) < 8:
        print(f'Warning: {year}-07-{day:02d} has {len(day_keys)} hourly maps')

    y_parts = []
    hour_parts = []
    lat_parts = []
    lon_parts = []
    valid_parts = []

    for hour_idx, key in enumerate(day_keys):
        sub = raw[key].loc[meta['mask'], ['Latitude', 'Longitude', 'ColumnAmountO3']].copy()
        vals = pd.to_numeric(sub['ColumnAmountO3'], errors='coerce').to_numpy(dtype=float)
        lat_vals = pd.to_numeric(sub['Latitude'], errors='coerce').to_numpy(dtype=float)
        lon_vals = pd.to_numeric(sub['Longitude'], errors='coerce').to_numpy(dtype=float)
        valid = np.isfinite(vals) & np.isfinite(lat_vals) & np.isfinite(lon_vals)
        y_parts.append(vals)
        hour_parts.append(np.full(vals.shape, hour_idx, dtype=int))
        lat_parts.append(lat_vals)
        lon_parts.append(lon_vals)
        valid_parts.append(valid)

    y_all = np.concatenate(y_parts)
    hour_all = np.concatenate(hour_parts)
    lat_all = np.concatenate(lat_parts)
    lon_all = np.concatenate(lon_parts)
    valid_all = np.concatenate(valid_parts)

    n_hours = len(day_keys)
    hour_design = np.zeros((valid_all.sum(), n_hours), dtype=float)
    hour_design[np.arange(valid_all.sum()), hour_all[valid_all]] = 1.0
    lat_mean = np.nanmean(lat_all[valid_all])
    lon_mean = np.nanmean(lon_all[valid_all])
    lat_center = lat_all[valid_all] - lat_mean
    lon_center = lon_all[valid_all] - lon_mean
    x = np.column_stack([hour_design, lat_center, lon_center])
    beta, *_ = np.linalg.lstsq(x, y_all[valid_all], rcond=None)

    grids = []
    variances = []
    fitted_means = []
    start = 0
    n_cells = len(meta['lat_idx'])
    for hour_idx, key in enumerate(day_keys):
        stop = start + n_cells
        vals = y_all[start:stop]
        lat_vals = lat_all[start:stop]
        lon_vals = lon_all[start:stop]
        valid = valid_all[start:stop]
        x_hour = np.zeros((n_cells, n_hours + 2), dtype=float)
        x_hour[:, hour_idx] = 1.0
        x_hour[:, n_hours] = lat_vals - lat_mean
        x_hour[:, n_hours + 1] = lon_vals - lon_mean
        mean_hat = x_hour @ beta
        resid = vals - mean_hat
        resid[~valid] = np.nan

        grid = np.full((len(meta['lats']), len(meta['lons'])), np.nan, dtype=float)
        grid[meta['lat_idx'], meta['lon_idx']] = resid
        grids.append(grid)
        variances.append(float(np.nanvar(resid, ddof=1)))
        fitted_means.append(float(np.nanmean(mean_hat[valid])))
        start = stop

    result = {
        'year': year,
        'day': day,
        'hour_keys': day_keys,
        'grids': grids,
        'residual_variances': np.asarray(variances, dtype=float),
        'ols_beta': beta,
        'fitted_hour_mean': np.asarray(fitted_means, dtype=float),
    }
    RESIDUAL_GRID_CACHE[cache_key] = result
    return result


In [ ]:
def shifted_semivariance(grid: np.ndarray, direction: str, offset: int) -> Tuple[float, int]:
    if offset == 0:
        raise ValueError('Zero spatial lag is excluded from pure spatial semivariograms.')
    if direction == 'lat':
        a = grid[:-offset, :]
        b = grid[offset:, :]
    elif direction == 'lon':
        a = grid[:, :-offset]
        b = grid[:, offset:]
    else:
        raise ValueError(direction)
    mask = np.isfinite(a) & np.isfinite(b)
    n = int(mask.sum())
    if n == 0:
        return np.nan, 0
    diff = a[mask] - b[mask]
    return float(0.5 * np.mean(diff * diff)), n


def compute_day_empirical_semivariogram(year: int, day: int):
    meta = build_grid_meta(year)
    residual = compute_day_residual_grids(year, day)
    grids = residual['grids']
    residual_variances = residual['residual_variances']

    lat_offsets = [int(round(float(lag) / meta['lat_step'])) for lag in LAT_LAGS]
    lon_offsets = [int(round(float(lag) / meta['lon_step'])) for lag in LON_LAGS]
    lat_hourly = []
    lon_hourly = []
    lat_counts = []
    lon_counts = []

    for h, grid in enumerate(grids[:8]):
        lat_vals, lat_ns = zip(*[shifted_semivariance(grid, 'lat', off) for off in lat_offsets])
        lon_vals, lon_ns = zip(*[shifted_semivariance(grid, 'lon', off) for off in lon_offsets])
        lat_vals = np.asarray(lat_vals, dtype=float)
        lon_vals = np.asarray(lon_vals, dtype=float)
        if NORMALIZE_EMPIRICAL:
            denom = residual_variances[h]
            if np.isfinite(denom) and denom > 0:
                lat_vals = lat_vals / denom
                lon_vals = lon_vals / denom
            else:
                lat_vals[:] = np.nan
                lon_vals[:] = np.nan
        lat_hourly.append(lat_vals)
        lon_hourly.append(lon_vals)
        lat_counts.append(lat_ns)
        lon_counts.append(lon_ns)

    lat_hourly = np.asarray(lat_hourly, dtype=float)
    lon_hourly = np.asarray(lon_hourly, dtype=float)
    lat_counts = np.asarray(lat_counts, dtype=int)
    lon_counts = np.asarray(lon_counts, dtype=int)

    return {
        'year': year,
        'day': day,
        'hour_keys': residual['hour_keys'],
        'lat_lags': LAT_LAGS,
        'lon_lags': LON_LAGS,
        'lat_hourly': lat_hourly,
        'lon_hourly': lon_hourly,
        'lat_mean': np.nanmean(lat_hourly, axis=0),
        'lon_mean': np.nanmean(lon_hourly, axis=0),
        'lat_counts': lat_counts,
        'lon_counts': lon_counts,
        'residual_variances': residual_variances,
        'mean_model': MEAN_MODEL,
    }


In [ ]:
MODEL_STYLE = {
    'nu=0.3 x1 nugget_free': {'label': 'x1 nu=0.3', 'color': '#d62728'},
    'nu=0.5 x1 nugget_free': {'label': 'x1 nu=0.5', 'color': '#1f77b4'},
}
BLUE_COLORS = plt.cm.Blues(np.linspace(0.35, 0.92, 8))


def matern_corr(scaled_distance: np.ndarray, smooth: float) -> np.ndarray:
    h = np.asarray(scaled_distance, dtype=float)
    out = np.ones_like(h, dtype=float)
    positive = h > 0
    if not np.any(positive):
        return out
    hp = h[positive]
    if np.isclose(smooth, 0.5):
        out[positive] = np.exp(-hp)
    else:
        coef = 1.0 / (2.0 ** (smooth - 1.0) * gamma_fn(smooth))
        vals = coef * (hp ** smooth) * kv(smooth, hp)
        out[positive] = vals
    out[~np.isfinite(out)] = 0.0
    return np.clip(out, -1.0, 1.0)


def fitted_spatial_semivariogram(lags: np.ndarray, row: pd.Series, direction: str) -> np.ndarray:
    sigmasq = float(row['sigma'])
    nugget = float(row['nugget'])
    smooth = float(row['smooth'])
    range_param = float(row['range_lat'] if direction == 'lat' else row['range_lon'])
    h = np.asarray(lags, dtype=float)
    scaled = np.abs(h) / max(range_param, 1e-12)
    corr = matern_corr(scaled, smooth)
    gamma = nugget + sigmasq * (1.0 - corr)
    if NORMALIZE_EMPIRICAL:
        sill = sigmasq + nugget
        gamma = gamma / sill if np.isfinite(sill) and sill > 0 else np.full_like(gamma, np.nan)
    return gamma


def estimate_rows_for_day(year: int, day: int) -> pd.DataFrame:
    return estimates.loc[estimates['year'].eq(year) & estimates['day_num'].eq(day)].copy()


def missing_models_for_day(day: int) -> list[str]:
    ok = set(estimate_rows_for_day(YEAR, day)['smooth'].round(6).tolist())
    return [f'nu={s:g}' for s in SMOOTH_SOURCES if round(float(s), 6) not in ok]


def summarize_curve_error(year: int, day: int, emp: dict):
    rows = []
    est_day = estimate_rows_for_day(year, day)
    for direction, lags, empirical_mean in [
        ('lat', emp['lat_lags'], emp['lat_mean']),
        ('lon', emp['lon_lags'], emp['lon_mean']),
    ]:
        for _, est in est_day.iterrows():
            fitted = fitted_spatial_semivariogram(np.asarray(lags), est, direction)
            valid = np.isfinite(empirical_mean) & np.isfinite(fitted)
            residual = empirical_mean[valid] - fitted[valid]
            rows.append({
                'year': year,
                'month': MONTH,
                'day_num': day,
                'direction': direction,
                'model': est['model'],
                'smooth': est['smooth'],
                'rmse': float(np.sqrt(np.mean(residual ** 2))) if residual.size else np.nan,
                'mae': float(np.mean(np.abs(residual))) if residual.size else np.nan,
                'bias_emp_minus_fit': float(np.mean(residual)) if residual.size else np.nan,
                'mean_model': MEAN_MODEL,
                'normalized': NORMALIZE_EMPIRICAL,
                'sigma': est['sigma'],
                'range_lat': est['range_lat'],
                'range_lon': est['range_lon'],
                'range_time': est['range_time'],
                'nugget': est['nugget'],
                'resolution_label': est['resolution_label'],
                'variant': est['variant'],
            })
    return rows


In [ ]:
def plot_day_semivariogram(year: int, day: int, emp: dict, out_dir: Path, save=True):
    fig, axes = plt.subplots(1, 2, figsize=(15.8, 6.4), constrained_layout=True)
    est_day = estimate_rows_for_day(year, day)

    panels = [
        ('lat', 'Latitude-direction residual raw semivariogram', emp['lat_lags'], emp['lat_hourly'], emp['lat_mean']),
        ('lon', 'Longitude-direction residual raw semivariogram', emp['lon_lags'], emp['lon_hourly'], emp['lon_mean']),
    ]

    for ax, (direction, title, lags, hourly, daily_mean) in zip(axes, panels):
        for h in range(min(8, hourly.shape[0])):
            ax.plot(lags, hourly[h], color=BLUE_COLORS[h], lw=1.25, alpha=0.86, label=f'Hour {h + 1}')
        ax.plot(lags, daily_mean, color='black', lw=3.1, label='Daily mean over 8 hours', zorder=5)

        first_lag = float(np.nanmin(lags))
        max_lag = float(np.nanmax(lags))
        dense_lags = np.linspace(first_lag, max_lag, 240)
        for _, est in est_day.iterrows():
            style = MODEL_STYLE.get(est['model'], {'label': est['model'], 'color': '0.25'})
            fitted = fitted_spatial_semivariogram(dense_lags, est, direction)
            ax.plot(dense_lags, fitted, color=style['color'], lw=2.6, label=f'{style["label"]} fitted t=0')

        ax.set_ylim(*Y_LIMIT)
        ax.set_xlim(0.0, max_lag)
        ax.set_xlabel(f'{direction} lag (degrees)')
        ax.set_ylabel('residual semivariance')
        ax.set_title(title)
        ax.grid(True, alpha=0.32)

    missing = missing_models_for_day(day)
    suffix = '' if not missing else f' | missing x1 params: {", ".join(missing)}'
    fig.suptitle(
        f'{year}-07-{day:02d}: residual raw hourly/daily spatial semivariogram vs implied-spatial x1 ST fits{suffix}',
        fontsize=15,
    )
    handles, labels = axes[1].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=5, fontsize=8.4, frameon=True, bbox_to_anchor=(0.5, -0.070))
    if save:
        out_path = out_dir / f'daily_spatial_semivariogram_residual_raw_implied_spatial_x1_smooth03_05_{year}_07_day{day:02d}.png'
        fig.savefig(out_path, bbox_inches='tight')
    return fig


In [ ]:
RUN_ALL = True
SHOW_EACH_DAY = False

all_empirical_rows = []
all_fit_rows = []
all_error_rows = []

if RUN_ALL:
    year_dir = OUTPUT_DIR / f'year_{YEAR}'
    daily_dir = year_dir / 'daily_semivariograms'
    year_dir.mkdir(parents=True, exist_ok=True)
    daily_dir.mkdir(parents=True, exist_ok=True)
    pdf_path = year_dir / f'daily_spatial_semivariogram_residual_raw_implied_spatial_x1_smooth03_05_{YEAR}_07_days01_10.pdf'
    print(f'\n=== Processing {YEAR}-07 days 01-10 ===')
    with PdfPages(pdf_path) as pdf:
        for day in DAYS:
            print(f'Processing {YEAR}-07-{day:02d}...')
            emp = compute_day_empirical_semivariogram(YEAR, day)
            for direction, lags, hourly, daily_mean, counts in [
                ('lat', emp['lat_lags'], emp['lat_hourly'], emp['lat_mean'], emp['lat_counts']),
                ('lon', emp['lon_lags'], emp['lon_hourly'], emp['lon_mean'], emp['lon_counts']),
            ]:
                for j, lag in enumerate(lags):
                    row = {
                        'year': YEAR,
                        'month': MONTH,
                        'day_num': day,
                        'direction': direction,
                        'lag': float(lag),
                        'daily_mean': float(daily_mean[j]) if np.isfinite(daily_mean[j]) else np.nan,
                        'mean_pair_count': float(np.nanmean(counts[:, j])),
                        'mean_model': MEAN_MODEL,
                        'normalized': NORMALIZE_EMPIRICAL,
                    }
                    for h in range(hourly.shape[0]):
                        row[f'hour_{h + 1}'] = float(hourly[h, j]) if np.isfinite(hourly[h, j]) else np.nan
                        row[f'n_pairs_hour_{h + 1}'] = int(counts[h, j])
                        row[f'residual_var_hour_{h + 1}'] = float(emp['residual_variances'][h]) if h < len(emp['residual_variances']) else np.nan
                    all_empirical_rows.append(row)

            est_day = estimate_rows_for_day(YEAR, day)
            for direction, lags in [('lat', emp['lat_lags']), ('lon', emp['lon_lags'])]:
                for _, est in est_day.iterrows():
                    fitted = fitted_spatial_semivariogram(np.asarray(lags), est, direction)
                    for lag, gamma_hat in zip(lags, fitted):
                        all_fit_rows.append({
                            'year': YEAR,
                            'month': MONTH,
                            'day_num': day,
                            'direction': direction,
                            'lag': float(lag),
                            'model': est['model'],
                            'smooth': est['smooth'],
                            'fitted_gamma_t0': float(gamma_hat),
                            'mean_model': MEAN_MODEL,
                            'normalized': NORMALIZE_EMPIRICAL,
                            'sigma': est['sigma'],
                            'range_lat': est['range_lat'],
                            'range_lon': est['range_lon'],
                            'range_time': est['range_time'],
                            'nugget': est['nugget'],
                            'resolution_label': est['resolution_label'],
                            'variant': est['variant'],
                        })
            all_error_rows.extend(summarize_curve_error(YEAR, day, emp))

            fig = plot_day_semivariogram(YEAR, day, emp, daily_dir, save=True)
            pdf.savefig(fig, bbox_inches='tight')
            if SHOW_EACH_DAY:
                plt.show()
            else:
                plt.close(fig)
    print(f'Saved {pdf_path}')

empirical_table = pd.DataFrame(all_empirical_rows).round(4)
fitted_table = pd.DataFrame(all_fit_rows).round(4)
error_table = pd.DataFrame(all_error_rows).round(4)

empirical_csv = OUTPUT_DIR / 'empirical_daily_spatial_semivariogram_residual_raw_implied_spatial_x1_2024_07_days01_10.csv'
fitted_csv = OUTPUT_DIR / 'fitted_t0_spatial_semivariogram_residual_raw_implied_spatial_x1_smooth03_05_2024_07_days01_10.csv'
error_csv = OUTPUT_DIR / 'spatial_semivariogram_residual_raw_curve_error_implied_spatial_x1_smooth03_05_2024_07_days01_10.csv'
empirical_table.to_csv(empirical_csv, index=False, float_format='%.4f')
fitted_table.to_csv(fitted_csv, index=False, float_format='%.4f')
error_table.to_csv(error_csv, index=False, float_format='%.4f')
print(f'Saved {empirical_csv}')
print(f'Saved {fitted_csv}')
print(f'Saved {error_csv}')

year_dir = OUTPUT_DIR / f'year_{YEAR}'
empirical_table.to_csv(year_dir / f'empirical_daily_spatial_semivariogram_residual_raw_implied_spatial_x1_{YEAR}_07_days01_10.csv', index=False, float_format='%.4f')
fitted_table.to_csv(year_dir / f'fitted_t0_spatial_semivariogram_residual_raw_implied_spatial_x1_smooth03_05_{YEAR}_07_days01_10.csv', index=False, float_format='%.4f')
error_table.to_csv(year_dir / f'spatial_semivariogram_residual_raw_curve_error_implied_spatial_x1_smooth03_05_{YEAR}_07_days01_10.csv', index=False, float_format='%.4f')

display(error_table.head(12))


In [ ]:
if 'error_table' in globals() and not error_table.empty:
    summary_by_direction = (
        error_table
        .groupby(['direction', 'model'], dropna=False)
        .agg(
            n=('rmse', 'size'),
            rmse_mean=('rmse', 'mean'),
            rmse_median=('rmse', 'median'),
            mae_mean=('mae', 'mean'),
            mae_median=('mae', 'median'),
            bias_mean=('bias_emp_minus_fit', 'mean'),
            bias_median=('bias_emp_minus_fit', 'median'),
        )
        .round(4)
        .reset_index()
    )
    summary_overall = (
        error_table
        .groupby(['model'], dropna=False)
        .agg(
            n=('rmse', 'size'),
            rmse_mean=('rmse', 'mean'),
            rmse_median=('rmse', 'median'),
            mae_mean=('mae', 'mean'),
            mae_median=('mae', 'median'),
            bias_mean=('bias_emp_minus_fit', 'mean'),
            bias_median=('bias_emp_minus_fit', 'median'),
        )
        .round(4)
        .reset_index()
    )
    summary_by_direction.to_csv(OUTPUT_DIR / 'curve_error_summary_by_direction_implied_spatial_x1_smooth03_05_2024_07_days01_10.csv', index=False)
    summary_overall.to_csv(OUTPUT_DIR / 'curve_error_summary_overall_implied_spatial_x1_smooth03_05_2024_07_days01_10.csv', index=False)
    display(summary_by_direction)
    display(summary_overall)
else:
    print('No curve-error rows available. Check x1 parameter status table above.')


## Notes

- Empirical curves are computed from residual maps after fitting `ColumnAmountO3 ~ hour fixed effects + latitude + longitude` separately for each day.
- Fitted curves use only `resolution_label == "x1"` and `variant == "nugget_free"` rows from the implied-spatial ST Vecchia output folders.
- The source folder name is `lag643`, and the notebook preserves `lag0_block_count`, `lag1_block_count`, and `lag2_block_count` in the parameter/status CSVs so the exact Vecchia pattern can be checked.
- If a smooth/day x1 parameter row is not `ok`, the empirical semivariogram is still plotted, but that smooth's fitted curve is omitted and marked in the figure title.
